# 06 · Prophet with regressors (Stage 2)
**Goal:** a second model family to compare with LightGBM and the baselines, on the same split and daytime test rows.

**Where to run.** Windows Smart App Control blocks Prophet's Stan executable on the project PC, so this notebook is run on **Google Colab**:
1. Locally, zip `src/` and `data/processed/` into `ojt_colab.zip` (keep the folder structure).
2. Open this notebook in Colab, run the first cell, and upload the zip.
3. At the end, download `reports/prophet_results.csv` and `reports/prophet_quantile_results.csv` into the project's `reports/` folder.

It runs unchanged on any machine where Prophet works.

**Setup**
- One Prophet model per (site, horizon h, weather mode). Flat trend, daily and yearly seasonality, no weekly seasonality (the sun has no weekday). Fitted on daytime training hours only.
- **Regressors** are taken from `build_features`, so they obey the same causality rules as LightGBM:
  - always: clear-sky GHI at t+h (deterministic);
  - `lagged` (realistic): CF and clear-sky index at origin t;
  - `forecast` (upper bound): actual GHI, temperature and clear-sky index at t+h, plus cloud cover where available.
- The 80 % uncertainty interval (`yhat_lower`, `yhat_upper`) gives P10 and P90.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    !pip -q install prophet pvlib
    from google.colab import files
    files.upload()                         # upload ojt_colab.zip
    !unzip -oq ojt_colab.zip -d /content/OJT_Project
    os.makedirs("/content/OJT_Project/notebooks", exist_ok=True)
    os.makedirs("/content/OJT_Project/reports/figures", exist_ok=True)
    os.chdir("/content/OJT_Project/notebooks")

In [ ]:
import logging, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").disabled = True
import numpy as np
import pandas as pd
from prophet import Prophet
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

from io_utils import ROOT, load_processed
from config import SITES
from features import build_features, chronological_split
from evaluation import all_baselines, evaluate, fit_clearsky_scale, interval_metrics, pinball, skill

HORIZONS = [1, 2, 3, 4, 5, 6, 24, 36, 48]
MODES = ["lagged", "forecast"]
REGRESSORS = {
    "lagged": ["ghi_clearsky", "cf_lag_1", "kt_at_origin"],
    "forecast": ["ghi_clearsky", "w_ghi", "w_temp_c", "w_clearsky_index", "w_cloud_cover_pct"],
}
data = {}
for s in ("primary", "site_a"):
    d = load_processed(s)
    d.attrs["tz"] = SITES[s]["tz"]
    data[s] = d

## 1. Fit and predict

In [ ]:
def prophet_frame(X, regs):
    f = X[regs].copy()
    f["ds"] = X.index.tz_localize(None)
    f["y"] = X["y"].to_numpy()
    return f.reset_index(drop=True)


def run_prophet(X, regs):
    train, test = chronological_split(X)
    regs = [r for r in regs if r in X.columns]
    day = train[(train["is_day"] == 1) & train["y"].notna()].dropna(subset=regs)
    m = Prophet(growth="flat", daily_seasonality=True, weekly_seasonality=False,
                yearly_seasonality=True, interval_width=0.8)
    for r in regs:
        m.add_regressor(r)
    m.fit(prophet_frame(day, regs))
    full = pd.concat([train, test]).dropna(subset=regs)
    fc = m.predict(prophet_frame(full, regs).drop(columns="y"))
    P = pd.DataFrame({"p10": fc["yhat_lower"].to_numpy(), "p50": fc["yhat"].to_numpy(),
                      "p90": fc["yhat_upper"].to_numpy()}, index=full.index)
    P = pd.DataFrame(np.sort(P.to_numpy(), axis=1).clip(0, 1.05), index=P.index, columns=P.columns)
    P.loc[full["is_day"] == 0] = 0.0
    return P.reindex(X.index)


preds = {}
for s, d in data.items():
    for mode in MODES:
        for h in HORIZONS:
            preds[(s, mode, h)] = run_prophet(build_features(d, h, weather_mode=mode), REGRESSORS[mode])
            print(f"{s:8s} {mode:8s} h={h:2d} done")

## 2. Point accuracy against the baselines (same daytime test rows)

In [ ]:
rows = []
for s, d in data.items():
    cut = d.index[int(len(d) * 0.8)]
    train = d[d.index < cut]
    mask = (d.index >= cut) & (d["is_day"] == 1)
    for h in HORIZONS:
        table = all_baselines(d, train, h, SITES[s]["tz"], fit_clearsky_scale(train))
        for mode in MODES:
            table[f"prophet_{mode}"] = preds[(s, mode, h)]["p50"]
        for model, r in evaluate(d, table, mask).iterrows():
            rows.append({"site": s, "horizon_h": h, "model": model, **r.to_dict()})

results = pd.DataFrame(rows)
results["RMSE_%cap"] = results["RMSE"] * 100
results["MAE_%cap"] = results["MAE"] * 100
is_base = ~results["model"].str.startswith("prophet")
ref = results[is_base].loc[results[is_base].groupby(["site", "horizon_h"])["RMSE"].idxmin()].set_index(["site", "horizon_h"])
results["best_baseline"] = [ref.loc[(r.site, r.horizon_h), "model"] for r in results.itertuples()]
results["skill_vs_best_baseline"] = [skill(r.RMSE, ref.loc[(r.site, r.horizon_h), "RMSE"]) for r in results.itertuples()]
results.pivot_table(index=["site", "model"], columns="horizon_h", values="RMSE_%cap").round(2)

In [ ]:
results[~is_base].pivot_table(index=["site", "model"], columns="horizon_h", values="skill_vs_best_baseline").round(3)

## 3. Uncertainty: P10–P90 interval

In [ ]:
qrows = []
for (s, mode, h), P in preds.items():
    d = data[s]
    cut = d.index[int(len(d) * 0.8)]
    ok = (d.index >= cut) & (d["is_day"] == 1) & d["capacity_factor"].notna() & P.notna().all(axis=1)
    y, P = d.loc[ok, "capacity_factor"], P[ok]
    qrows.append({"site": s, "mode": mode, "horizon_h": h, **interval_metrics(y, P["p10"], P["p90"]),
                  **{f"pinball_{k}": pinball(y, P[k], a) for k, a in {"p10": .1, "p50": .5, "p90": .9}.items()}})
qres = pd.DataFrame(qrows)
qres.pivot_table(index=["site", "mode"], columns="horizon_h", values="coverage").round(3)

In [ ]:
os.makedirs(ROOT / "reports", exist_ok=True)
results.round(5).to_csv(ROOT / "reports" / "prophet_results.csv", index=False)
qres.round(5).to_csv(ROOT / "reports" / "prophet_quantile_results.csv", index=False)
if "google.colab" in sys.modules:
    files.download(str(ROOT / "reports" / "prophet_results.csv"))
    files.download(str(ROOT / "reports" / "prophet_quantile_results.csv"))

## Findings
_(filled in after the Colab run)_